In [110]:
import pandas as pd
from pathlib import Path

In [111]:
df = pd.read_csv(
    "/home/ulises/Descargas/datos/ventas/ventas_enero.csv",
    parse_dates=["fecha"]
)

"""
No pude leer la carea directamente asi que ingrese la ruta,
ruego descomente la linea siguiente para que la utilice ya que lo de arriba solo funciona en mi propio laptop
"""
# df = pd.read_csv("ventas_enero.csv", parse_dates=["fecha"])


'\nNo pude leer la carea directamente asi que ingrese la ruta,\nruego descomente la linea siguiente para que la utilice ya que lo de arriba solo funciona en mi propio laptop\n'

In [112]:
df.head()

,fecha,producto,vendedor,region,cantidad,precio_unitario,total
0,2024-01-01,Hub USB,Matias Soto,Poniente,1,18000,18000
1,2024-01-02,Mouse,Matias Soto,Norte,4,22000,88000
2,2024-01-03,Auriculares,Matias Soto,Poniente,2,68000,136000
3,2024-01-04,Hub USB,Matias Soto,Poniente,5,18000,90000
4,2024-01-05,Teclado,Paula Mena,Norte,4,45000,180000


In [113]:
df.shape

(31, 7)

In [114]:
df.dtypes

fecha              datetime64[us]
producto                      str
vendedor                      str
region                        str
cantidad                    int64
precio_unitario             int64
total                       int64
dtype: object

In [115]:
df.isnull().sum()

fecha              0
producto           0
vendedor           1
region             0
cantidad           0
precio_unitario    0
total              0
dtype: int64

In [116]:
df.describe()

,fecha,cantidad,precio_unitario,total
count,31,31.000000,31.000000,3.100000e+01
mean,2024-01-16 00:00:00,2.838710,73483.870968,1.851290e+05
min,2024-01-01 00:00:00,1.000000,18000.000000,1.800000e+04
25%,2024-01-08 12:00:00,2.000000,33500.000000,7.500000e+04
50%,2024-01-16 00:00:00,3.000000,55000.000000,1.350000e+05
75%,2024-01-23 12:00:00,4.000000,75000.000000,1.920000e+05
max,2024-01-31 00:00:00,5.000000,290000.000000,1.160000e+06
std,NaN,1.392762,75017.718337,2.134432e+05


In [117]:
df_vendedores = df[df["vendedor"].notna()]

In [118]:
df_vendedores.shape[0]

30

In [119]:
carpeta = Path("/home/ulises/Descargas/datos/ventas")

# El mismo problema de antes ruego comente la linea de arriba y descomente la de abajo para que funcione en su laptop
#carpeta = Path("datos/ventas")

In [120]:
archivos = sorted(carpeta.glob("*.csv"))
print("Archivos CSV encontrados:", len(archivos))
dfs = []
total_filas_individuales = 0

for archivo in archivos:
    df = pd.read_csv(archivo, parse_dates=["fecha"])

    # Extraer el mes desde el nombre del archivo
    df["mes"] = archivo.stem.replace("ventas_", "")

    total_filas_individuales += len(df)

    dfs.append(df)

# Consolidar
df_consolidado = pd.concat(dfs, ignore_index=True)

print("\nFilas consolidadas:", len(df_consolidado))
print("Suma filas individuales:", total_filas_individuales)

Archivos CSV encontrados: 12

Filas consolidadas: 397
Suma filas individuales: 397


In [121]:
filas_consolidadas = len(df_consolidado)

validacion = filas_consolidadas == total_filas_individuales

print("\n¿Validación exitosa?:", "Si" if validacion else "No")


¿Validación exitosa?: Si


In [122]:
df_consolidado.columns.tolist()

['fecha',
 'producto',
 'vendedor',
 'region',
 'cantidad',
 'precio_unitario',
 'total',
 'mes']

In [123]:
df_consolidado.head()

,fecha,producto,vendedor,region,cantidad,precio_unitario,total,mes
0,2024-04-01,Monitor,Camila Vega,Oriente,2,290000,580000,abril
1,2024-04-02,Disco SSD,Valeria Rios,Norte,2,75000,150000,abril
2,2024-04-03,Teclado,Valeria Rios,Poniente,5,45000,225000,abril
3,2024-04-04,Auriculares,Diego Parra,Poniente,4,68000,272000,abril
4,2024-04-05,Webcam,Camila Vega,Norte,2,55000,110000,abril


In [124]:
resumen_mensual = (
    df_consolidado
    .groupby("mes")["total"]
    .sum()
    .reset_index()
)

print(resumen_mensual)

           mes     total
0        abril  17987000
1       agosto  16307000
2    diciembre  28508000
3        enero   5739000
4      febrero  12895000
5        julio  24528000
6        junio  22156000
7        marzo  22336000
8         mayo  19499000
9    noviembre  15973000
10     octubre  12831000
11  septiembre  17044000


In [125]:
resumen_productos = (
    df_consolidado
    .groupby("producto")["cantidad"]
    .sum()
    .reset_index()
    .sort_values("cantidad", ascending=False)
)

resumen_productos

producto_mas_vendido = resumen_productos.iloc[0]
producto_mas_vendido

producto    Webcam
cantidad       176
Name: 7, dtype: object

In [126]:
mes_mayor_ingreso = resumen_mensual.loc[
    resumen_mensual["total"].idxmax()
]

mes_mayor_ingreso

mes      diciembre
total     28508000
Name: 2, dtype: object

In [127]:
promedio_venta = df_consolidado["total"].mean()


promedio_venta

np.float64(543584.3828715365)

In [128]:
Path("/home/ulises/Descargas/datos/salida").mkdir(
#Path("datos/salida").mkdir(
    parents=True,
    exist_ok=True
)

In [129]:
df_consolidado.to_csv(
    "/home/ulises/Descargas/datos/salida/ventas_2024_consolidado.csv",
    #"datos/salida/ventas_2024_consolidado.csv"
    index=False,
    encoding="utf-8"
)

In [130]:
with pd.ExcelWriter(
    "/home/ulises/Descargas/datos/salida/reporte_anual_2024.xlsx",
    #"datos/salida/reporte_anual_2024.xlsx",
    engine="openpyxl"
) as writer:

    df_consolidado.to_excel(
        writer,
        sheet_name="Datos_Completos",
        index=False
    )

    resumen_mensual.to_excel(
        writer,
        sheet_name="Resumen_Mensual",
        index=False
    )

    resumen_productos.to_excel(
        writer,
        sheet_name="Resumen_Productos",
        index=False
    )

In [131]:
with pd.ExcelFile(
    "/home/ulises/Descargas/datos/salida/reporte_anual_2024.xlsx"
    #"datos/salida/reporte_anual_2024.xlsx"
) as libro:

    print(libro.sheet_names)

['Datos_Completos', 'Resumen_Mensual', 'Resumen_Productos']


In [132]:
#pd.read_csv("datos/ventas/ventas_enero.CSV") --> hay que agregar que el "csv" se encuentra en mayusculas eh
#pd.read_csv("/home/ulises/Descargas/datos/ventas/ventas_enero.CSV")
ruta = Path("/home/ulises/Descargas/datos/ventas/ventas_enero.csv")
#ruta = Path("datos/ventas/ventas_enero.csv")

if ruta.exists():
    df = pd.read_csv(ruta)
    print("Archivo cargado correctamente")
else:
    print("Archivo no encontrado:", ruta.resolve())

Archivo cargado correctamente


In [133]:
"""
carpeta_erronea = Path("datos/ventas_2025")

dfs = [pd.read_csv(a) for a in carpeta_erronea.glob("*.csv")]

pd.concat(dfs)
"""
carpeta_erronea = Path("/home/ulises/Descargas/datos/ventas_2025")
#carpeta_erronea = Path("datos/ventas_2025")

archivos = list(carpeta_erronea.glob("*.csv"))

if len(archivos) == 0:
    print("No se encontraron archivos en:", carpeta_erronea)
else:
    resultado = pd.concat(
        [pd.read_csv(a) for a in archivos],
        ignore_index=True
    )

    print(resultado.head())

No se encontraron archivos en: /home/ulises/Descargas/datos/ventas_2025


In [134]:
#pd.read_csv("/home/ulises/Descargas/datos/archivo_cp1252.csv")

In [135]:
pd.read_csv(
    "/home/ulises/Descargas/datos/archivo_cp1252.csv",
    #"datos/archivo_cp1252.csv"
    encoding="cp1252"
)

,nombre,ciudad
0,José,Valparaíso
1,María,Concepción


In [136]:
dfs = []

for archivo in sorted(Path("datos/ventas").glob("*.csv")):
    df = pd.read_csv(archivo)
    dfs.append(df)

df["mes"] = archivo.stem

In [137]:
dfs = []

for archivo in sorted(Path("/home/ulises/Descargas/datos/ventas").glob("*.csv")):
#for archivo in sorted(Path("datos/ventas").glob("*.csv")):

    df = pd.read_csv(archivo)

    # Asignar el mes dentro del ciclo
    df["mes"] = archivo.stem.replace("ventas_", "")

    dfs.append(df)

df_consolidado = pd.concat(
    dfs,
    ignore_index=True
)

print(df_consolidado.head())

        fecha     producto      vendedor    region  cantidad  precio_unitario  \
0  2024-04-01      Monitor   Camila Vega   Oriente         2           290000   
1  2024-04-02    Disco SSD  Valeria Rios     Norte         2            75000   
2  2024-04-03      Teclado  Valeria Rios  Poniente         5            45000   
3  2024-04-04  Auriculares   Diego Parra  Poniente         4            68000   
4  2024-04-05       Webcam   Camila Vega     Norte         2            55000   

    total    mes  
0  580000  abril  
1  150000  abril  
2  225000  abril  
3  272000  abril  
4  110000  abril  
